# sensor_node_v1 — Battery-Powered BLE Environmental Sensor Node

ESP32-C3-MINI-1 + BME280 (temp/hum/press) + VEML7700 (light) + MCP73831 LiPo charger + AP2112K 3.3V LDO + USB-C charge port.

In [ ]:
import pathlib
import hw_toolkit as hw
from hw_toolkit.iface import Power, I2C
from hw_toolkit.parts import Buck

board = hw.Board("sensor_node_v1")
print(f"Board created: {board.project_id}")

## Load-first: MCU — ESP32-C3-MINI-1

In [ ]:
# MCU: ESP32-C3-MINI-1 — primary load, defines power rail and I2C bus
mcu = board.module(
    id="mcu",
    category="mcu_module",
    mpn="ESP32-C3-MINI-1-H4",
    package="LCC-58",
    price_usd=3.50,
    manufacturer="Espressif",
)
mcu_vdd = Power(hv=mcu.pin("VDD"), lv=mcu.pin("GND"), voltage=3.3)
mcu.expose(
    vdd=mcu_vdd,
    i2c0=I2C(
        scl=mcu.pin("IO5"),
        sda=mcu.pin("IO4"),
        frequency=400_000,
        reference=mcu_vdd,
    ),
)
mcu.show()  # inline render of MCU subsystem

## BME280 — Temp / Humidity / Pressure Sensor

In [ ]:
# BME280: I2C environmental sensor
bme = board.module(
    id="bme",
    category="environmental_sensor",
    mpn="BME280",
    package="LGA-8",
    price_usd=3.80,
    manufacturer="Bosch",
)
bme_vdd = Power(hv=bme.pin("VDD"), lv=bme.pin("GND"), voltage=3.3)
bme.expose(
    vdd=bme_vdd,
    i2c=I2C(
        scl=bme.pin("SCK"),
        sda=bme.pin("SDI"),
        frequency=400_000,
        reference=bme_vdd,
    ),
)
bme.show()  # inline render of BME280 subsystem

## VEML7700 — Ambient Light Sensor

In [ ]:
# VEML7700: I2C ambient light sensor (0–120 klux, 90dB dynamic range)
veml = board.module(
    id="veml",
    category="light_sensor",
    mpn="VEML7700-TT",
    package="ODFN-6",
    price_usd=1.50,
    manufacturer="Vishay",
)
veml_vdd = Power(hv=veml.pin("VDD"), lv=veml.pin("GND"), voltage=3.3)
veml.expose(
    vdd=veml_vdd,
    i2c=I2C(
        scl=veml.pin("SCL"),
        sda=veml.pin("SDA"),
        frequency=400_000,
        reference=veml_vdd,
    ),
)
veml.show()  # inline render of VEML7700 subsystem

## AP2112K-3.3 — 3.3V LDO

In [ ]:
# AP2112K-3.3: 600mA LDO, 55µA Iq — low-Iq for battery life
# Pinout: VIN, GND, VOUT, EN, BYP
ldo = board.module(
    id="ldo",
    category="ldo_regulator",
    mpn="AP2112K-3.3TRG1",
    package="SOT-25",
    price_usd=0.45,
    manufacturer="Diodes Inc",
)
ldo_in = Power(hv=ldo.pin("VIN"), lv=ldo.pin("GND"), voltage=4.2)
ldo_out = Power(hv=ldo.pin("VOUT"), lv=ldo.pin("GND"), voltage=3.3)
ldo.expose(
    power_in=ldo_in,
    power_out=ldo_out,
)
ldo.show()  # inline render of LDO subsystem

## MCP73831 — LiPo Charger

In [ ]:
# MCP73831T: 500mA linear LiPo charger, PROG pin sets charge current
# Pinout: VDD (VBUS in), VSS (GND), VBAT (battery out), PROG, STAT
chgr = board.module(
    id="chgr",
    category="battery_charger",
    mpn="MCP73831T-2ACI/OT",
    package="SOT-23-5",
    price_usd=0.75,
    manufacturer="Microchip",
)
chgr_vbus = Power(hv=chgr.pin("VDD"), lv=chgr.pin("VSS"), voltage=5.0)
chgr_vbat = Power(hv=chgr.pin("VBAT"), lv=chgr.pin("VSS"), voltage=4.2)
chgr.expose(
    power_in=chgr_vbus,
    power_out=chgr_vbat,
)
chgr.show()  # inline render of charger subsystem

## USB-C Connector (charge-only)

In [ ]:
# GCT USB4085: USB-C connector, charge-only (no data lines used)
# Pins: VBUS, GND, CC1, CC2, D+, D-
conn = board.module(
    id="conn",
    category="usb_connector",
    mpn="USB4085-GF-A",
    package="SMD",
    price_usd=0.85,
    manufacturer="GCT",
)
conn_usb = Power(hv=conn.pin("VBUS"), lv=conn.pin("GND"), voltage=5.0)
conn.expose(power_out=conn_usb)
conn.show()  # inline render of USB-C connector subsystem

## Status LED + User Button

In [ ]:
# Status LED — green 0603, Vf=2.2V
led = board.module(
    id="led",
    category="led",
    mpn="150060GS75000",
    package="0603",
    price_usd=0.12,
    manufacturer="Wurth",
)
led.show()

In [ ]:
# User button — active-low tactile switch
btn = board.module(
    id="btn",
    category="switch",
    mpn="PTS841GKSMSMTRLFS",
    package="4.6x3.0mm",
    price_usd=0.35,
    manufacturer="C&K",
)
btn.show()

## Passives

In [ ]:
# LED series resistor: 33Ω (3.3V - 2.2V Vf) / 33Ω ≈ 33mA
r_led = board.resistor("R_LED", "33", package="0603")
# Charger PROG: Iprog = 1000/R → 2kΩ = 500mA
r_prog = board.resistor("R_PROG", "2k", package="0603")
# Button pull-up: 10k to 3.3V
r_btn = board.resistor("R_BTN", "10k", package="0603")
# LDO EN pull-up: 100k from VBAT to EN (always-on)
r_en = board.resistor("R_EN", "100k", package="0603")
# USB CC pull-downs: 5.1k to advertise 5V/500mA
r_cc1 = board.resistor("R_CC1", "5k1", package="0402")
r_cc2 = board.resistor("R_CC2", "5k1", package="0402")
# Decoupling caps
c_vbus = board.capacitor("C_VBUS", "4u7", package="0805")
c_vbat = board.capacitor("C_VBAT", "10u", package="0805")
c_vdd1 = board.capacitor("C_VDD1", "100n", package="0402")
c_vdd2 = board.capacitor("C_VDD2", "10u",  package="0805")
print("Passives added:",
      r_led.id, r_prog.id, r_btn.id, r_en.id,
      r_cc1.id, r_cc2.id,
      c_vbus.id, c_vbat.id, c_vdd1.id, c_vdd2.id)

## Wire: Power Path — USB-C → Charger → LDO → 3.3V Rail

Typed `Power.connect_to()` for the primary power path. Decoupling caps via legacy `net +=` API.

In [ ]:
# VBUS: USB connector → charger VDD (both 5V Power)
conn.power_out.connect_to(chgr.power_in)

# Find the created VBUS + GND nets and add decoupling cap
vbus_net = board._find_net_with_member(("conn", "VBUS"))
gnd_net  = board._find_net_with_member(("conn", "GND"))
vbus_net += "c_vbus.POS"
gnd_net  += "c_vbus.NEG"

print("VBUS net:", vbus_net.id, vbus_net.members)
print("GND  net:", gnd_net.id,  gnd_net.members)

In [ ]:
# VBAT: charger VBAT → LDO VIN (both 4.2V Power)
chgr.power_out.connect_to(ldo.power_in)

vbat_net = board._find_net_with_member(("chgr", "VBAT"))
vbat_net += "c_vbat.POS"
gnd_net  += "c_vbat.NEG"

# LDO EN always-on pull-up: VBAT → 100k → EN pin
vbat_net += "r_en.A"
ldo_en_net = board.signal("ldo_en", protocol="gpio")
ldo_en_net += "r_en.B", "ldo.EN"

print("VBAT net:", vbat_net.id, vbat_net.members)
print("LDO EN:", ldo_en_net.members)

In [ ]:
# V3V3: LDO VOUT → MCU VDD (typed, 3.3V → 3.3V)
ldo.power_out.connect_to(mcu.vdd)

# Find the 3.3V rail net and add remaining consumers
v3v3_net = board._find_net_with_member(("ldo", "VOUT"))
v3v3_net += "bme.VDD", "veml.VDD"
v3v3_net += "c_vdd1.POS", "c_vdd2.POS"
v3v3_net += "r_btn.A"          # button pull-up high side
v3v3_net += "bme.CSB"          # BME280 CSB → VDD for I2C mode

# All GNDs: MCU, BME, VEML, LDO
gnd_net += "mcu.GND", "bme.GND", "veml.GND"
gnd_net += "ldo.GND"
gnd_net += "c_vdd1.NEG", "c_vdd2.NEG"
gnd_net += "bme.SDO"           # BME280 SDO → GND → I2C addr 0x76

print("V3V3 net:", v3v3_net.id, "count:", len(v3v3_net.members))
print("GND  net:", gnd_net.id,  "count:", len(gnd_net.members))

## Wire: I²C Bus + Pull-ups

In [ ]:
# Wire MCU I2C to BME280 (typed, both 400kHz 3.3V)
mcu.i2c0.connect_to(bme.i2c)

# Add VEML7700 to the same SCL/SDA nets via legacy += (extend existing)
scl_net = board._find_net_with_member(("mcu", "IO5"))
sda_net = board._find_net_with_member(("mcu", "IO4"))
scl_net += "veml.SCL"
sda_net += "veml.SDA"

# Add 4.7k pull-ups (once, on the MCU's I2C reference power)
r_scl, r_sda = mcu.i2c0.add_pulls(value="4k7")

print("I2C pull-ups:", r_scl.id, r_sda.id)
print("SCL net:", scl_net.members)
print("SDA net:", sda_net.members)

## Wire: GPIO — LED, Button, Charger PROG, USB CC

In [ ]:
# LED: MCU IO6 → R_LED (33Ω) → LED anode → LED cathode → GND
led_gpio = board.signal("led_gpio", protocol="gpio")
led_gpio += "mcu.IO6", "r_led.A"

led_anode_net = board.signal("led_anode", protocol="analog")
led_anode_net += "r_led.B", "led.A"

gnd_net += "led.K"

# Button: MCU IO7 ← (R_BTN pull-up to v3v3) → button → GND
btn_gpio = board.signal("btn_gpio", protocol="gpio")
btn_gpio += "mcu.IO7", "r_btn.B", "btn.A"

gnd_net += "btn.B"

# Charger PROG: PROG pin → 2kΩ → GND
chg_prog_net = board.signal("chg_prog", protocol="analog")
chg_prog_net += "chgr.PROG", "r_prog.A"
gnd_net += "r_prog.B"

# USB CC pull-downs (5.1k → GND advertises 5V/500mA)
cc1_net = board.signal("usb_cc1", protocol="analog")
cc1_net += "conn.CC1", "r_cc1.A"
cc2_net = board.signal("usb_cc2", protocol="analog")
cc2_net += "conn.CC2", "r_cc2.A"
gnd_net += "r_cc1.B", "r_cc2.B"

print("LED GPIO:", led_gpio.members)
print("BTN GPIO:", btn_gpio.members)
print("CHG PROG:", chg_prog_net.members)
print("USB CC1/CC2:", cc1_net.members, cc2_net.members)

In [ ]:
# NC pins: charger STAT (open-drain, optional), USB D+/D- (charge-only)
stat_nc = board.nc("chgr_stat_nc")
stat_nc += "chgr.STAT"

dp_nc = board.nc("usb_dp_nc")
dp_nc += "conn.DP"

dm_nc = board.nc("usb_dm_nc")
dm_nc += "conn.DM"

print("NC sentinels wired")
print("GND net total members:", len(gnd_net.members))

## Board Summary

In [ ]:
print(board.summary())

## Full Schematic (board.show())

All nets now have ≥2 members — safe to render the complete schematic.

In [ ]:
board.show()

## ERC Check

In [ ]:
board.check_erc(expected_codes=(
    "pin_not_connected",          # MCP73831 STAT NC, USB-C SBU/data pins
    "lib_symbol_issues",          # hwagent lib synthesized at runtime
    "pin_to_pin",                 # LDO EN pull-up, BME CSB/SDO ties
    "power_pin_not_driven",       # connector power pins without PWR_FLAG
    "unconnected_wire_endpoint",  # synthesized wire-layout artifact
))
print("ERC passed")

## Export KiCad Project + Final Render

In [ ]:
import pathlib

out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/sensor_node_v1/sensor_node_v1.zip")
board.export_kicad(out, unzip=True)

print(f"Exported: {out}")
print(f"Zip exists: {out.exists()}")

# Final full schematic render
board.show()